In [3]:
import duckdb
import pandas as pd
import plotly.express as px
import numpy as np

db = duckdb.connect("../../data/retail.db", read_only=True)
db.sql("SET search_path TO mart;")
db.sql("""SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'mart'""")

┌─────────────────┐
│   table_name    │
│     varchar     │
├─────────────────┤
│ customerdim     │
│ productdim      │
│ retail_enriched │
└─────────────────┘

# Customer Segmentation (RFM)

Marketing vill förstå vilka kunder som driver intäkter och hur kundbasen skiljer sig i beteende.

- Vilka kundsegment står för majoriteten av intäkterna?
- Hur skiljer sig köpbeteendet mellan olika typer av kunder?
- Vilka kunder är aktiva, och vilka visar tecken på att falla bort?

Segmentering baserad på:
- Recency
- Frequency
- Monetary


    Notering: 
Orders från staging.top_anomalies med anomaly_type = 'anomaly_customer' har filtrerats bort. Dessa representerar stora manuella justeringar, plattformsavgifter eller ovanligt stora kunder som snedvrider analysen.

In [2]:
# Antal unika registrerade användare och unika gästorders
reg_n_guest = db.sql("""
SELECT
COUNT(DISTINCT customerid) AS n_customers,
COUNT(DISTINCT CASE WHEN is_missing_customerid =1 THEN invoiceno END) AS n_guest_orders
FROM retail_enriched
""").df()
print(reg_n_guest)

   n_customers  n_guest_orders
0         4372            3710


In [3]:
# Totala intäkter uppdelat på registrerade kunder och gästorders. Exkl anomalikunder
customer_rev = db.sql("""
       SELECT 
       CASE 
        WHEN is_missing_customerid = 0 THEN 'Registered'
        ELSE 'Guest'
       END as customer_type,
       ROUND(SUM(total_price), 2) as revenue
       FROM retail_enriched  
       WHERE total_price > 0 
       AND is_product = 1
       AND (customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       ) OR customerid IS NULL)
       GROUP BY customer_type

       UNION ALL

       SELECT 
       'Total' as customer_type,
       ROUND(SUM(total_price), 2) as revenue
       FROM retail_enriched
       WHERE total_price > 0 
       AND is_product = 1
       AND (customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       ) OR customerid IS NULL)
       """).df()
print(customer_rev)

  customer_type     revenue
0    Registered  8449765.23
1         Guest  1520635.91
2         Total  9970401.14


In [4]:
# Isolerar Registered och Guest. Plottar sedan i pajgraf.
df_rev_plot = customer_rev[customer_rev['customer_type'] != "Total"]
pie_rev = px.pie(df_rev_plot,
       values='revenue',
       names='customer_type',
       title='Registered customers and Guests share of revenue')
pie_rev.write_html("../HTML/cust_rev_pie.html",
                   include_plotlyjs='cdn')

Tidsperiod: 2010-12 to 2011-12 

Unika registrerade användare och unika gästorders.

    Registrerade kunder: 4372
    Gästorders: 3710

Totala intäkter: 10255973

    Registrerade kunder: 84,7% av total
    Gäster: 15,3% av total

In [5]:
# Jämför returmått - Registrerad vs Gäst
customer_guest_return = db.sql("""
       SELECT 
                      
       CASE 
        WHEN is_missing_customerid = 0 THEN 'Registered'
        ELSE 'Guest'
       END as customer_type,
                      
       ROUND(COUNT(DISTINCT CASE WHEN is_return_line = 1 THEN invoiceno END) * 1.0 / NULLIF(COUNT(DISTINCT CASE WHEN is_credit_invoice = 0 THEN invoiceno END), 0), 4) as return_rate_order,
       ROUND(SUM(CASE WHEN total_price < 0 THEN ABS(total_price) END) / NULLIF(SUM(CASE WHEN total_price > 0 THEN total_price END), 0), 4) as return_rate_value


       FROM retail_enriched  
       WHERE is_product = 1
       AND (customerid NOT IN (
       SELECT DISTINCT customerid 
       FROM staging.top_anomalies 
       WHERE anomaly_type = 'anomaly_customer'
       ) OR customerid IS NULL)
       GROUP BY customer_type
                      
       """).df()
print(customer_guest_return)

  customer_type  return_rate_order  return_rate_value
0    Registered             0.1879             0.0283
1         Guest             0.0234             0.0242


In [6]:
# Jämför korgar - Registrerad vs Gäst
customer_guest_basket = db.sql("""
                               WITH basket_items AS (
                               
                                   SELECT 

                                   invoiceno,

                                   CASE 
                                   WHEN is_missing_customerid = 0 THEN 'Registered'
                                   ELSE 'Guest'
                                   END as customer_type,

                                   COUNT(DISTINCT stockcode) as n_items,
                                   SUM(total_price) as basket_value

                                   FROM retail_enriched
                                   WHERE total_price > 0
                                   AND is_product = 1 
                                   AND (customerid NOT IN (
                                   SELECT DISTINCT customerid 
                                   FROM staging.top_anomalies 
                                   WHERE anomaly_type = 'anomaly_customer'
                                   ) OR customerid IS NULL) 
                                   GROUP BY invoiceno, customer_type
                               )

       SELECT 

       customer_type,                
       ROUND(AVG(basket_value), 2) as avg_basket_value,
       ROUND(MEDIAN(basket_value), 2) as median_basket_value,
       ROUND(AVG(n_items), 2) as avg_n_items,
       ROUND(MEDIAN(n_items), 2) as median_n_items

       FROM basket_items
       GROUP BY customer_type

       """).df()
print(customer_guest_basket)


  customer_type  avg_basket_value  median_basket_value  avg_n_items  \
0    Registered            459.00               301.29        20.96   
1         Guest           1106.72               361.15        95.24   

   median_n_items  
0            15.0  
1            38.0  


In [7]:
# Plottar korgvärden - registrerad vs gäst
fig_cgb = px.bar(customer_guest_basket,
       x='customer_type',
       y=['avg_basket_value','median_basket_value'],
       title='Basket Values',
       barmode='group')

fig_cgb.update_layout(
    xaxis_title='Customer Type',
    yaxis_title='Value',
    legend_title='Metric'
)
fig_cgb.write_html("../HTML/cust_basket_value.html",
                   include_plotlyjs='cdn')

In [8]:
# Plottar unika varor per korg - registrerad vs gues
fig_cgi = px.bar(customer_guest_basket,
                 x='customer_type',
                 y=['avg_n_items', 'median_n_items'],
                 title='Basket items',
                 barmode='group')
fig_cgi.update_layout(
    xaxis_title='Customer Type',
    yaxis_title='Number of items',
    legend_title='Metric'
)

fig_cgi.write_html("../HTML/cust_basket_items.html",
                   include_plotlyjs='cdn')

fig_cgi.show()

    Intäkter drivs av återkommande kunder, inte enstaka köp.

Ett problem i denna jämförelse är att vi antar att varje gästorder är en unik gäst, när det faktiskt skulle kunna vara ett par återkommande kunder som inte är registrerade, men det kan vi inte se. 

Vad vi kan se är däremot att gäster tycks spendera i genomsnitt mer per köp än vad våra registrerade kunder gör. De handlar bredare med betydligt mer variation i korgarna per order, men de handlar också betydligt mer sällan än våra registrerade kunder.

Gästernas totala köp står däremot endast för 15,3% av totala intäkter. Det är de registrerade kundernas återkommande köp som driver våra intäkter. Att konvertera gäster till mer återkommande kunder bör därav ses som en möjlighet att öka intäkter, inte per order, utan totalt sett på lång sikt.

De skillnader som kan observeras i returmåtten är troligtvis kopplade till köpfrekvens snarare än faktisk produktmissnöje. Fler orders innebär fler tillfällen där en enskild vara inte uppfyller förväntningarna eller har pushats fel. Utan data på returanledningar kan det inte bekräftas.

I och med att registrerade kunder står för hela 84,7% av intäkterna så ska vi titta lite närmare på hur de beter sig och hur de skiljer sig åt inom gruppen. 

In [9]:
df_reg_customer = db.sql("""
                         SELECT
                         total_spend,
                         days_since_last_purchase,
                         total_orders
                         FROM customerdim
                         WHERE (customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              ) OR customerid IS NULL)
                         ORDER BY days_since_last_purchase
                         """).df()
df_reg_customer[['total_spend','days_since_last_purchase', 'total_orders']].describe()

,total_spend,days_since_last_purchase,total_orders
count,4335.000000,4336.0,4369.000000
mean,1984.229826,91.988238,4.241245
std,8530.068505,99.960643,7.687636
min,3.750000,0.0,0.000000
25%,306.455000,17.0,1.000000
50%,668.430000,50.0,2.000000
75%,1655.795000,141.0,5.000000
max,280206.020000,373.0,210.000000


In [10]:
# Distribution av data (total_spend, total_orders, days_since_last_purchase)
# Underlag för thresholds/cutoffs
fig_spend = px.histogram(df_reg_customer,
             x='total_spend',
             title='Total Spend distribution'
             )
fig_spend.update_traces(xbins=dict(size=250))
fig_spend.show()
px.histogram(df_reg_customer,
             x='total_orders',
             title='Total Orders distribution').show()
px.histogram(df_reg_customer,
             x='days_since_last_purchase',
             title='Days since last purchase distribution').show()

In [11]:
# Distribution av (total_spend, total_orders, days_since_last_purchase).
# Hanterar skew och extrema värden genom att Logaritmera total_spend och total_orders
plot_reg_customer = df_reg_customer.copy()
plot_reg_customer['log_spend'] = np.log1p(plot_reg_customer['total_spend'])
plot_reg_customer['log_order'] = np.log1p(plot_reg_customer['total_orders'])

px.histogram(plot_reg_customer,
             x='log_spend',
             title='log Total Spend distribution').show()
px.histogram(plot_reg_customer,
             x='log_order',
             title='log Total Orders distribution').show()
px.histogram(plot_reg_customer,
             x='days_since_last_purchase',
             title='Days since last purchase distribution').show()
# Överväg Jenks Natural Breaks?

In [12]:
db.sql("""SELECT 
    PERCENTILE_CONT(0.33) WITHIN GROUP (ORDER BY total_orders) as freq_low,
    PERCENTILE_CONT(0.67) WITHIN GROUP (ORDER BY total_orders) as freq_high,
    PERCENTILE_CONT(0.33) WITHIN GROUP (ORDER BY total_spend) as spend_low,
    PERCENTILE_CONT(0.67) WITHIN GROUP (ORDER BY total_spend) as spend_high
    FROM mart.customerdim
       """)

┌──────────┬───────────┬────────────────────┬────────────────────┐
│ freq_low │ freq_high │     spend_low      │     spend_high     │
│  double  │  double   │       double       │       double       │
├──────────┼───────────┼────────────────────┼────────────────────┤
│      1.0 │       4.0 │ 384.10100000000006 │ 1212.8021999999999 │
└──────────┴───────────┴────────────────────┴────────────────────┘

Att kunna identifiera kundgrupper som driver värde är centralt i skapandet av riktade insatser. Det avgör inte bara vad för typ av insats vi pushar utan också när och hur ofta.

Genom att titta på kundmått som total_orders, days_since_last_purchase och total_spend kan vi gruppera kunder efter beteende snarare än demografi, och därmed prioritera rätt insats till rätt grupp.

Utifrån hur distributionen för dessa mått ser ut så sätter vi brytpunkter för hur vi ska kategorisera värdena. I days_since_last_purchase går det att urskilja två brytpunkter där rate of change tydligt avtar. Dessa brytpunkter, vid 39 och 89 dagar hjälper oss att definiera de nivåer, low, mid och high, som vi kommer att använda oss av för att dela in kunderna i olika segmenten.

För days_since_last_purchase blir Low: 0-39, Mid: 40-89, High: 90+

För total_spend och total_orders är valet inte lika självklart då datan i båda måtten är väldigt skeva. På grund av detta så delar vi upp dessa i percentiler, 33e och 67e, vilket sammanfaller väl med valet att använda tre nivåer i days_since_last_purchase.

För total_spend blir Low: >384, Mid: 384-1212 , High: 1212+ 
För total_orders blir Low: 1, Mid: 2-4, High: 5+

(Nivåer satta vid 33e och 67e percentilen, exempelvis "High", är relativt 
till kundbasen, inte ett absolut högt belopp. De speglar relativa skillnader i kundbasen.)

Vi applicerar sedan dessa nivåer för att tilldela kunderna poäng, 1 ,2 och 3, där högre är bättre.

In [13]:
# RFM-scoring
customer_scoring = db.sql("""
                          SELECT
                          Customerid,
                          total_spend,
                          total_orders,
                          days_since_last_purchase,

                          CASE 
                            WHEN total_spend >= 1212 THEN 3
                            WHEN total_spend >= 384 THEN 2
                            ELSE 1 
                          END as monetary_score,
                          
                          CASE
                            WHEN total_orders >= 5 THEN 3
                            WHEN total_orders >= 2 THEN 2
                            ELSE 1 
                          END as frequency_score,

                          CASE
                            WHEN days_since_last_purchase <= 39 THEN 3 
                            WHEN days_since_last_purchase <= 89 THEN 2
                            ELSE 1
                          END as recency_score

                          FROM customerdim
                          WHERE (customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              ) OR customerid IS NULL) 
                          """).df()
customer_scoring[['monetary_score','recency_score','frequency_score']].apply(lambda col: col.value_counts())

,monetary_score,recency_score,frequency_score
1,1464,1480,1526
2,1474,956,1728
3,1431,1933,1115


In [14]:
# Ser över distribution av scoring
print(customer_scoring[['monetary_score']].value_counts().sort_index())
print(customer_scoring[['frequency_score']].value_counts().sort_index())
print(customer_scoring[['recency_score']].value_counts().sort_index())
# Det lägre antalet i recency_score: 2 kan tolkas som övergångsfas mellan aktiv och inaktiv, där kunder sällan befinner sig en längre tid.

monetary_score
1                 1464
2                 1474
3                 1431
Name: count, dtype: int64
frequency_score
1                  1526
2                  1728
3                  1115
Name: count, dtype: int64
recency_score
1                1480
2                 956
3                1933
Name: count, dtype: int64


Vi delar sedan upp kunderna i sju grupper. Champions, Loyal, Big spender, New, At Risk, Opportunity och sist inaktiva/övrigt. Kunderna kvalificerar sig i dessa grupper beroende på de poäng vi tilldelat dem.

    Champions:   R = 3    F = 3    M = 3  
    Big Spender: R = 1-3  F = 1-3  M = 3   
    Loyal:       R = 2-3  F = 3    M = 1-3  
    At Risk:     R = 1-2  F = 1-3  M = 2-3 
    New:         R = 3    F = 1    M = 1-3 
    Opportunity: R = 2    F = 2    M = 1-3  
    Inaktiva/Övrigt      

I och med att "Big Spender" och "Loyal" kan överlappa så kommer kunder som kvalificerar sig som "Big Spender" att prioriteras för det segmentet. Denna prioritering baseras på de olika typer av insatser som är bäst lämpade för de två segmenten. Big Spenders som driver intäkter på kort sikt motiverar potentiellt direkta och mer flexibla insatser, medan Loyal per design är ett mer långsiktigt relationsbygge. 

In [15]:
# Segmentlogik och segmentering
customer_segment = db.sql("""
                          WITH customer_scoring as(
                          SELECT
                          *,

                          CASE 
                            WHEN total_spend >= 1212 THEN 3
                            WHEN total_spend >= 384 THEN 2
                            ELSE 1 
                          END as m_score,
                          
                          CASE
                            WHEN total_orders >= 5 THEN 3
                            WHEN total_orders >= 2 THEN 2
                            ELSE 1 
                          END as f_score,

                          CASE
                            WHEN days_since_last_purchase <= 39 THEN 3 
                            WHEN days_since_last_purchase <= 89 THEN 2
                            ELSE 1
                          END as r_score

                          FROM customerdim
                          WHERE (customerid NOT IN (
                              SELECT DISTINCT customerid 
                              FROM staging.top_anomalies 
                              WHERE anomaly_type = 'anomaly_customer'
                              ) OR customerid IS NULL) 
                          )

                          SELECT
                          *,

                          CASE
                            WHEN r_score = 3 AND f_score = 3 AND m_score = 3 THEN 'champion'
 
                            WHEN r_score >= 1 AND f_score >= 1 AND m_score = 3 THEN 'big_spender'
                    
                            WHEN r_score >= 2 AND f_score = 3 AND m_score >= 1 THEN 'loyal'
                        
                            WHEN r_score <= 2 AND f_score >= 1 AND m_score >= 2 THEN 'at_risk'
                         
                            WHEN r_score = 3 AND f_score = 1 AND m_score >= 1 THEN 'new'
                       
                            WHEN r_score >= 2 AND f_score = 2 AND m_score >= 1 THEN 'opportunity'
                          
                            ELSE 'inactive_other'
                          
                          END as segment

                          FROM customer_scoring
                          
                          """).df()

In [16]:
# Ser över de kunder vars poäng eventuellt inte ledde till segmentering. Gör även eventuella justeringar i cutoffs.
print(customer_segment[customer_segment['segment'].isna()][['r_score','f_score','m_score']].value_counts())
# Ser sedan över distribution av segment.
print(customer_segment.groupby('segment').size().sort_values(ascending=False))
# Lågt antal i "loyal" på grund av överlapp med big_spender som har prio. Champions klassas tekniskt sett också som loyal.

Series([], Name: count, dtype: int64)
segment
inactive_other    1037
at_risk            857
champion           781
big_spender        650
opportunity        642
new                280
loyal              122
dtype: int64


In [17]:
db.register('customer_segment_tbl', customer_segment)

In [18]:
# Segmentöversikt intäktsmått
segment_overview = db.sql("""
       WITH spending_stats AS (
       SELECT 
       segment,
       COUNT(*) as customers,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct_customer,
       ROUND(SUM(total_spend), 2) as spending,
       ROUND(SUM(total_spend) * 100.0 / SUM(SUM(total_spend)) OVER (), 2) as spending_share,
       ROUND(SUM(total_spend) / COUNT(*)) as avg_customer_spend
       FROM customer_segment_tbl 
       GROUP BY segment
       )
       SELECT 
       *,
       ROUND(spending_share / pct_customer, 2) as spend_metric
       FROM spending_stats
       ORDER BY spend_metric desc
       """).df()
print(segment_overview)

          segment  customers  pct_customer    spending  spending_share  \
0        champion        781          17.9  5621237.39           65.35   
1     big_spender        650          14.9  1624520.52           18.89   
2           loyal        122           2.8   111294.98            1.29   
3         at_risk        857          19.6   578957.21            6.73   
4     opportunity        642          14.7   373676.33            4.34   
5             new        280           6.4    81029.54            0.94   
6  inactive_other       1037          23.7   210920.32            2.45   

   avg_customer_spend  spend_metric  
0              7197.0          3.65  
1              2499.0          1.27  
2               912.0          0.46  
3               676.0          0.34  
4               582.0          0.30  
5               289.0          0.15  
6               203.0          0.10  


In [19]:
rev_v_pop = px.bar(segment_overview,
       x='segment',
       y=['spending_share', 'pct_customer'],
       barmode='group',
       title='Intäktsandel vs kundandel per segment')
rev_v_pop.write_html("../HTML/cust_rev_v_pop.html",
                     include_plotlyjs='cdn')


Segmentöversikten ger oss en klar bild över vilka kunder som faktiskt driver intäkter.
Intäkterna är tydligt koncentrerade till en liten del av kundbasen. Champions (17,9% av kunderna) står för 65,35% av omsättningen, och tillsammans med “Big Spenders” (14,9%) genererar de 84,2% av intäkterna från registrerade kunder.

Att en liten del av kundbasen står för en betydande del av intäkterna skapar både möjlighet för bättre riktade retention-insatser, men utgör också en risk på grund av intäktskoncentrationen.

För att kunna konkretisera dessa möjligheter och risker bättre så tittar vi lite djupare på segmentens köpbeteenden.

In [20]:
segment_behavior = db.sql("""
       WITH segment_stats AS(
       SELECT
       cs.segment,

       ROUND(MEDIAN(CASE WHEN re.is_credit_invoice = 0 THEN re.basket_value END), 2) as median_basket_value,

       ROUND(COUNT(DISTINCT CASE WHEN re.is_return_line = 1 THEN re.invoiceno END) * 1.0 / NULLIF(COUNT(DISTINCT CASE WHEN re.is_credit_invoice = 0 THEN re.invoiceno END), 0), 4) as return_rate_order,
       
       ROUND(SUM(CASE WHEN re.total_price < 0 THEN ABS(re.total_price) END) / NULLIF(SUM(CASE WHEN re.total_price > 0 THEN re.total_price END), 0), 4) as return_rate_value,

       ROUND(AVG(cs.avg_days_between_orders), 2) as avg_d_between_orders,
       
       ROUND(MEDIAN(cs.days_since_last_purchase), 0) as median_days_since_purchase,

       MODE(cs.favorite_purchase_day) as favorite_purchase_day,

       MODE(cs.top_product) as favorite_product

       FROM customer_segment_tbl cs
       JOIN retail_enriched re ON re.customerid=cs.customerid
       WHERE re.is_product = 1
       AND cs.customerid NOT IN (
              SELECT DISTINCT customerid 
              FROM staging.top_anomalies 
              WHERE anomaly_type = 'anomaly_customer'
              ) 
       GROUP BY cs.segment
       ) 

       SELECT
       ss.*,
       pd.description as favorite_product_description
       FROM segment_stats ss
       LEFT JOIN productdim pd ON ss.favorite_product = pd.stockcode
       ORDER BY ss.median_days_since_purchase 
       """).df()

,segment,median_basket_value,return_rate_order,return_rate_value,avg_d_between_orders,median_days_since_purchase,favorite_purchase_day,favorite_product,favorite_product_description
0,champion,431.11,0.2020,0.0270,27.33,7.0,Wednesday,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,loyal,180.51,0.1220,0.0248,55.54,18.0,Sunday,85123A,WHITE HANGING HEART T-LIGHT HOLDER
2,opportunity,252.23,0.1249,0.0175,100.21,19.0,Sunday,21034,REX CASH+CARRY JUMBO SHOPPER
3,new,329.36,0.1036,0.0115,NaN,22.0,Monday,22082,RIBBON REEL STRIPES DESIGN
4,big_spender,609.01,0.2228,0.0374,65.47,53.0,Thursday,85123A,WHITE HANGING HEART T-LIGHT HOLDER
5,at_risk,360.00,0.1801,0.0232,87.43,99.0,Thursday,85123A,WHITE HANGING HEART T-LIGHT HOLDER
6,inactive_other,207.73,0.1499,0.0363,71.31,185.0,Thursday,85123A,WHITE HANGING HEART T-LIGHT HOLDER


In [21]:
fig_days_between = px.bar(segment_behavior,
                  x='segment',
                  y='avg_d_between_orders',
                  title='Genomsnittligt antal dagar mellan orders per segment')
fig_days_between.write_html("../HTML/cust_days_between_orders.html",
                            include_plotlyjs='cdn')

Vi såg att intäkterna är tydligt koncentrerade till en mindre grupp kunder. Vad för typ av beteende som driver intäkter skiljer sig däremot åt mellan segmenten. 

"Champion", som är verksamhetens mest värdefulla kundgrupp, kännetecknas av hög köpfrekvens (avg_d_between_orders 27.33) och medelstora ordervärden (median_basket_value: 431,11).

I kontrast handlar "Big spender" mindre frekvent (avg_d_between_orders 65.47) men till betydligt högre ordervärden (median_basket_value: 609.01)

Dessa skillnader i beteende ger oss en tydligare bild av vilka typer av kampanjer eller insatser som bäst lämpar sig för respektive kundgrupp.

För att förstå om ett segment är aktivt eller på väg att falla bort så kan genomsnittlig tid mellan köp sättas i relation till median tid sedan senaste köp. Detta ger oss en indikativ bild om kundbeteendet ligger i linje med historiska mönster. 

”Big spender” har en median på 53 dagar sedan senaste köp, vilket ligger inom ramen för deras historiska köpfrekvens (65,47) och indikerar att gruppen fortfarande är aktiv.

”At Risk”-segmentet visar däremot att median dagar sedan senaste köp (99) har passerat deras historiska köpfrekvens (87.43), vilket signalerar att kunderna inom detta segment håller på att bli inaktiva.

Detta gör att kunder inom segmentet är relevanta för retention-insatser, då de ännu inte är helt förlorade men visar tydliga tecken på minskad aktivitet.

”Loyal”, som delar vissa attribut med ”Champion” och ”Big Spender” , fångar en liten grupp av frekvent återkommande kunder som inte riktigt når upp till top-segmenten på grund av deras relativt låga ordervärden (180,51). Givet deras tydligt aktiva köpbeteende bör relationer med denna kundgrupp vårdas över tid för att successivt öka ordervärden.

Lika viktigt som långsiktiga relationer är välkomnandet av nya kunder. Segmentet "New" som representerar de kunder som endast har lagt en order står för en väldigt liten del av intäkterna. Segmentet sticker däremot ut med väldigt låga returmått (return rate order: 0,1036, return rate value:0,0115) och skulle kunna tolkas som att förväntningarna möts väl, vilket är ett positivt tecken för konvertering.

Välkomnandet och det långsiktiga relationsbyggandet handlar om att etablera kunder och hjälpa dem hitta rätt. "Opportunity" består av kunder som fortfarande är aktiva, men inte riktigt har etablerat ett tydligt köpmönster. Precis som segmentnamnet antyder representerar detta kunder med potential att utvecklas mot antingen "Loyal" eller "Big Spender".



    Notering
avg_days_between_orders för inactive_other bör tolkas med försiktighet 
då majoriteten av segmentet saknar tillräckligt med 
köphistorik för ett meningsfullt day_between-mått.

Baserat på insikterna kring de olika kundsegmenten, där 32,8% av kundbasen driver 84,2% av intäkterna, är riktade insatser för enskilda segment mer effektiva än breda och generella kampanjer. 
Nedan följer ett par rekommendationer som grundas i identifierade segmentbeteenden.  

 
    Rekommendationer

* VIP-program riktat mot Champions och Big Spender. Tier-lösning med både spend och frekvens som mått för att pusha Champions på spend och Big Spender på frekvens.

* Implementera alert-trigger när kund rör sig mot "At Risk" samt identifiera befintliga At Risk-kunder med historiskt högt värde som prioriterade mål i retention-kampanjer.

* Frekvensbaserade belöningar som matchar "Loyal" köpbeteende och uppmuntrar till högre ordervärde.

* Onboarding-program för att konvertera guests, som vi såg spenderar mycket, till registrerade användare. Introducera eventuella tidsbegränsade förmåner såsom fri frakt och rabatt på ytterligare köp under onboardingperioden. Detta sänker tröskeln för ett andra köp och uppmuntrar till att etablera ett köpbeteende. 

* I "Opportunity" finns potential för kunder att nå "Loyal" och "Big spender". Med en nudge i form av rabatt på 3e köpet, eller fri frakt under begränsad period, kan vi pusha dessa i rätt riktning.
    * Individuell segmentering inom Opportunity för att rikta mer precis insats och pusha kunder åt det håll med minst friktion. Exempelvis:
        - Hög recency + Låg spend -> Spend-fokuserade erbjudanden
        - Hög recency + Låg frekvens -> Incitament för återbesök
    
    


# Begränsningar
upd: 2026-04-13
* avg_days_between_orders för Inactive/Övrigt bör tolkas med försiktighet då
        majoriteten av segmentet saknar tillräcklig köphistorik för ett meningsfullt mått.

* Analysen baseras på ett ögonblicksvärde och fångar inte rörelseriktning på individnivå. Med längre tidsserie eller uppföljande mätpunkter skulle det gå att avgöra hur kunder rör sig över segment,eller om kunder rör sig mot högre eller lägre aktivitet över tid.
    - Skulle öka precisionen i exempelvis retention-triggers.